In [ ]:
///- --package .
open sm'_operators

#### page (test)

In [ ]:
///- --test static

open testing

In [1]:
union kind =
    | Article
    | Release

type author = { name : string; handle : string }

type entry =
    {
        slug : string
        title : string
        date : string
        kind : kind
        authors : list author
        blurb : string
        blocks : list post.block
    }

inl kind_label (k : kind) : string =
    match k with
    | Article => ""
    | Release => "Release"

inl rec join_names (authors : list author) : string =
    match authors with
    | Nil => ""
    | Cons (a, Nil) => html.escape a.name
    | Cons (a, rest) => html.escape a.name ++# ", " ++# join_names rest

inl rec render_blocks (section : i32) (blocks : list post.block) : string =
    match blocks with
    | Nil => ""
    | Cons (post.Heading text, rest) =>
        "<h2 id=\"section-" ++#? section ++# "\">" ++# html.escape text ++# "</h2>" ++# render_blocks (section + 1) rest
    | Cons (b, rest) => post.render_block b ++# render_blocks section rest

inl rec toc_items (section : i32) (blocks : list post.block) : string =
    match blocks with
    | Nil => ""
    | Cons (post.Heading text, rest) =>
        "<li><a href=\"#section-"
        ++#? section
        ++# "\">"
        ++# html.escape text
        ++# "</a></li>"
        ++# toc_items (section + 1) rest
    | Cons (_, rest) => toc_items section rest

inl table_of_contents (blocks : list post.block) : string =
    inl items = toc_items 1 blocks
    if items = "" then "" else "<nav class=\"toc\"><p>On this page</p><ol>" ++# items ++# "</ol></nav>"

inl render_entry (e : entry) : string =
    "<article><h1 class=\"post-title\">" ++# html.escape e.title ++# "</h1>"
    ++# "<p class=\"byline\">" ++# join_names e.authors ++# " &middot; " ++# html.escape e.date ++# "</p>"
    ++# table_of_contents e.blocks ++# render_blocks 1 e.blocks ++# "</article>"

inl sample () : entry =
    {
        slug = "sample"
        title = "Sample Entry"
        date = "2026-10-17"
        kind = Article
        authors = Cons ({ name = "i574n"; handle = "i574n" }, Nil)
        blurb = "A fixture for the engine's own tests."
        blocks = Cons (post.Heading "The Matrix", Cons (post.Rule, Nil))
    }

#### an_article_has_no_type_label (test)

In [2]:
///- --test

kind_label Article |> _assert_eq ""

__assert_eq { expected =  }

#### the_byline_lists_authors_and_date (test)

In [3]:
///- --test

render_entry (sample ()) |> sm'.contains "<p class=\"byline\">i574n &middot; 2026-10-17</p>" |> _assert_eq true

__assert_eq { expected = true }

#### blocks_render_in_order (test)

In [4]:
///- --test

render_entry (sample ())
|> sm'.contains "<h2 id=\"section-1\">The Matrix</h2><hr class=\"section-rule\">"
|> _assert_eq true

__assert_eq { expected = true }

#### the_table_of_contents_links_every_heading (test)

In [5]:
///- --test

table_of_contents (Cons (post.Heading "A", Cons (post.Paragraph "x", Cons (post.Heading "B", Nil))))
|> _assert_eq (
       "<nav class=\"toc\"><p>On this page</p><ol><li><a href=\"#section-1\">A</a></li><li><a "
       ^^ "href=\"#section-2\">B</a></li></ol></nav>"
   )

__assert_eq { expected = <nav class="toc"><p>On this page</p><ol><li><a href="#section-1">A</a></li><li><a href="#section-2">B</a></li></ol></nav> }

#### a_post_without_headings_has_no_table_of_contents (test)

In [6]:
///- --test

table_of_contents (Cons (post.Paragraph "x", Nil)) |> _assert_eq ""

__assert_eq { expected =  }